# EfficientNetV2-B0 baseline — evaluation metrics

This notebook loads a saved baseline checkpoint from training (`src/train_baseline_efficientnet.py`) and reports classification metrics on the **same validation split** stored in the checkpoint.

**Outputs from training** (by default):
- `outputs/baseline_efficientnet/<timestamp>/checkpoint.pt` — model weights + hyperparameters + train/val indices
- `outputs/baseline_efficientnet/<timestamp>/metrics.json` — accuracy, balanced accuracy, macro F1, per-class metrics, confusion matrix

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# Project root = parent of notebooks/
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from baseline_metrics import (
    LESION_CLASS_NAMES,
    collect_predictions,
    compute_classification_metrics,
)
from data_process import SkinLesionDataset, NUM_LESION_TYPES
from train_baseline_efficientnet import (
    EfficientNetV2Baseline,
    build_transforms,
    get_device,
)

/opt/anaconda3/envs/gan/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Point to a saved run

Set `CHECKPOINT` to your `checkpoint.pt` path (or leave `None` to pick the latest under `outputs/baseline_efficientnet/`).

In [2]:
CHECKPOINT = None  # e.g. ROOT / "outputs/baseline_efficientnet/20260130_120000/checkpoint.pt"

if CHECKPOINT is None:
    base = ROOT / "outputs" / "baseline_efficientnet"
    runs = sorted(base.glob("*/checkpoint.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not runs:
        raise FileNotFoundError(f"No checkpoint found under {base}. Train with: python src/train_baseline_efficientnet.py")
    CHECKPOINT = runs[0]

CHECKPOINT = Path(CHECKPOINT)
print("Using checkpoint:", CHECKPOINT)

Using checkpoint: /Users/hoangnamtran/Desktop/Code/SkinLesionBiasReduction/outputs/baseline_efficientnet/20260330_195351/checkpoint.pt


In [3]:
device = get_device(None)
print("Device:", device)

ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
args_ns = ckpt.get("args", {})
pretrained = bool(args_ns.get("pretrained", True)) and not bool(args_ns.get("no_pretrained", False))
dropout = float(args_ns.get("dropout", 0.3))
image_size = int(args_ns.get("image_size", 64))
csv_path = ROOT / args_ns.get("csv_path", "dataset/fitzpatrick17k_cleaned.csv")
image_dir = ROOT / args_ns.get("image_dir", "dataset/images")
val_indices = ckpt["val_indices"]

model = EfficientNetV2Baseline(
    num_classes=NUM_LESION_TYPES,
    pretrained=pretrained,
    dropout=dropout,
).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Loaded model; image_size:", image_size)

Device: mps


Loaded model; image_size: 64


In [ ]:
_, val_transform = build_transforms(image_size)
val_dataset = SkinLesionDataset(
    csv_path=str(csv_path),
    image_dir=str(image_dir),
    transform=val_transform,
)
val_subset = Subset(val_dataset, val_indices)
val_loader = DataLoader(
    val_subset,
    batch_size=int(args_ns.get("batch_size", 32)),
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

y_true, y_pred, y_prob, skin_tones = collect_predictions(model, val_loader, device)
metrics = compute_classification_metrics(y_true, y_pred)

print("Accuracy:", round(metrics["accuracy"], 4))
print("Balanced accuracy:", round(metrics["balanced_accuracy"], 4))
print("Macro F1:", round(metrics["macro_f1"], 4))
print("Weighted F1:", round(metrics["weighted_f1"], 4))
print()
print(metrics["classification_report_str"])

In [ ]:
cm = np.array(metrics["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xticklabels=LESION_CLASS_NAMES,
    yticklabels=LESION_CLASS_NAMES,
    ylabel="True label",
    xlabel="Predicted label",
    title="Confusion matrix (validation set)",
)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j, i, format(cm[i, j], "d"),
            ha="center", va="center",
            color="white" if cm[i, j] > thresh else "black",
        )
fig.tight_layout()
plt.show()

## Optional: compare with `metrics.json` from the same run

Training writes the same numbers to `metrics.json` next to the checkpoint.

In [ ]:
json_path = CHECKPOINT.parent / "metrics.json"
if json_path.exists():
    with open(json_path, encoding="utf-8") as f:
        saved = json.load(f)
    cls = saved.get("classification", saved)  # support both new nested + old flat schemas
    print("From metrics.json — accuracy:", cls["accuracy"], "balanced:", cls["balanced_accuracy"])
else:
    print("No metrics.json beside checkpoint.")